# PubMed Abstract Sentence Classification

Clean companion notebook for the refactored PyTorch baseline.

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report

from pubmed_sentence_classification import (
    AbstractSentenceClassifier,
    create_dataloader,
    evaluate,
    load_splits,
    prepare_splits,
    train,
)

In [ ]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
elif PROJECT_DIR.name != "01_pubmed_sentence_classification":
    PROJECT_DIR = PROJECT_DIR / "01_pubmed_sentence_classification"
DATA_DIR = PROJECT_DIR / "data" / "raw" / "PubMed_20k_RCT"
MODEL_PATH = PROJECT_DIR / "models" / "pubmed_sentence_classifier.pt"

splits = load_splits(DATA_DIR)
datasets, word2idx, label2idx = prepare_splits(splits, min_frequency=2)
idx2label = {idx: label for label, idx in label2idx.items()}

train_loader = create_dataloader(datasets["train"], word2idx, batch_size=64, shuffle=True)
validation_loader = create_dataloader(datasets["validation"], word2idx, batch_size=64)
test_loader = create_dataloader(datasets["test"], word2idx, batch_size=64)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AbstractSentenceClassifier(
    vocab_size=len(word2idx),
    num_classes=len(label2idx),
    pad_idx=word2idx["<pad>"],
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
history = train(model, train_loader, validation_loader, criterion, optimizer, device, epochs=15, checkpoint_path=MODEL_PATH)

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
metrics = evaluate(model, test_loader, criterion, device)
target_names = [idx2label[idx] for idx in sorted(idx2label)]
print(metrics)
print(classification_report(metrics["labels"], metrics["predictions"], target_names=target_names, zero_division=0))